# 🏢 企业内部 API Agent — 从零构建教程

## 📖 你将学到什么

本教程将**从零开始**，手把手教你构建一个能够调用企业内部 API 的 AI Agent。
每一步你都会**亲手写代码**，最终完成一个完整的项目。

完成后你将掌握：

| 知识点 | 对应步骤 |
|--------|----------|
| Function Calling 原理 | Step 2 |
| 如何设计工具基类和注册表 | Step 1-2 |
| 如何编写一个可调用的 API 工具 | Step 2 |
| 提示词工程基础 | Step 3 |
| OpenAI 兼容协议的 LLM 封装 | Step 4 |
| ReAct 推理循环（Agent 核心） | Step 5 |
| 多工具并行执行 | Step 5 |
| 如何扩展新工具 | Step 6 |

## 🔑 使用的技术

- **Python 3.10+** (异步编程 `async/await`)
- **LLM**: deepseek-v4-flash（OpenAI 兼容协议）
- **HTTP 客户端**: httpx（异步）
- **API 端点**: `https://token-plan.cn-beijing.maas.aliyuncs.com/compatible-mode/v1`

> 💡 **无需 API Key**：本教程内置模拟模式，没有 API Key 也能运行完整示例。
> 如果你有 API Key，可以设置环境变量 `DEEPSEEK_API_KEY=你的key` 使用真实 LLM。

---

## 📦 前置准备

### 1. 安装依赖

In [1]:
# 安装依赖（如果已安装可跳过）
!pip install openai httpx nest_asyncio 2>&1 | tail -3

# 安装完成后，验证是否成功
import openai, httpx
print(f"✅ openai {openai.__version__} 已安装")
print(f"✅ httpx {httpx.__version__} 已安装")

'tail' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


✅ openai 2.38.0 已安装
✅ httpx 0.28.1 已安装


### 2. 启动 Mock API Server（可选）

本教程配套了一个 **Mock API Server**，提供 4 个模拟的公司内部 API 端点。
如果你想体验**真实的 HTTP 调用**（而非模拟模式），请打开一个新终端运行：

```bash
cd /path/to/notebook
python mock_api_server.py
```

**如果你不启动 Mock Server 也没关系**——教程代码会检测连通性，自动降级为模拟模式。

### 3. 创建项目目录

我们先创建项目的基本目录结构：

In [2]:
import os

# 创建项目目录结构
dirs = [
    "company_agent",
    "company_agent/tools",
    "company_agent/llm",
    "company_agent/prompts",
    "company_agent/agent",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("✅ 项目目录创建完成")
print("📁 目录结构:")
for root, dirnames, filenames in os.walk("company_agent"):
    level = root.replace("company_agent", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}📂 {os.path.basename(root)}/")

✅ 项目目录创建完成
📁 目录结构:
📂 company_agent/
  📂 agent/
    📂 __pycache__/
  📂 llm/
    📂 __pycache__/
  📂 prompts/
    📂 __pycache__/
  📂 tools/
    📂 __pycache__/
  📂 __pycache__/


> 🎯 **目标确认**：你现在有了一个空的 `company_agent/` 项目目录。
> 接下来的每个 Step，我们都会一步步往里面填充代码。

---

## 🏗️ Step 1: 工具基类 — 理解设计模式

### 💡 核心概念：为什么要用抽象基类？

我们希望每个工具（员工查询、项目查询……）都有**统一的接口**，这样 Agent 可以
用统一的方式调用任何工具。抽象基类（ABC）强制要求子类实现特定的方法，避免遗漏。

```text
  ┌─────────────────┐
  │    BaseTool     │  ← 抽象基类（定义接口）
  │  - name         │
  │  - description  │
  │  - parameters   │
  │  - __call__     │
  └────────┬────────┘
           │ 继承
     ┌─────┴─────┐
  GetEmployees  GetProjects  ...  ← 具体工具类
```

### 动手：编写 `tools/base.py`

In [3]:
# 创建 __init__.py（使 tools/ 成为 Python 包）
import os

os.makedirs("company_agent/tools", exist_ok=True)
with open("company_agent/tools/__init__.py", "w", encoding="utf-8") as f:
    f.write("""from .base import BaseTool, ToolRegistry

__all__ = ["BaseTool", "ToolRegistry"]
""")
print("✅ company_agent/tools/__init__.py 已创建")

✅ company_agent/tools/__init__.py 已创建


In [4]:
with open("company_agent/tools/base.py", "w", encoding="utf-8") as f:
    f.write('''
"""
工具基类与注册表 — 所有工具的统一接口
"""
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional


class BaseTool(ABC):
    """工具基类 — 所有工具必须继承此类"""

    @property
    @abstractmethod
    def name(self) -> str:
        """工具的唯一标识（例如 'get_employees'）"""
        pass

    @property
    @abstractmethod
    def description(self) -> str:
        """工具的功能描述（会告诉 LLM，帮助它决定何时调用此工具）"""
        pass

    @property
    @abstractmethod
    def parameters(self) -> Dict[str, Any]:
        """参数定义，JSON Schema 格式（会告诉 LLM 可以传什么参数）"""
        pass

    @abstractmethod
    async def __call__(self, **kwargs) -> str:
        """执行工具逻辑，返回结果字符串"""
        pass

    def to_openai_tool(self) -> Dict[str, Any]:
        """将工具转换为 OpenAI Function Calling 格式"""
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": self.parameters,
            },
        }


class ToolRegistry:
    """工具注册表 — 管理所有可用工具"""

    def __init__(self):
        self._tools: Dict[str, BaseTool] = {}

    def register(self, tool: BaseTool):
        """注册一个工具"""
        self._tools[tool.name] = tool

    def get(self, name: str) -> Optional[BaseTool]:
        """根据名称获取工具"""
        return self._tools.get(name)

    def get_all(self) -> List[BaseTool]:
        """获取所有已注册的工具"""
        return list(self._tools.values())

    def to_openai_tools(self) -> List[Dict[str, Any]]:
        """将所有工具批量转换为 OpenAI 格式"""
        return [tool.to_openai_tool() for tool in self._tools.values()]

    def __contains__(self, name: str) -> bool:
        return name in self._tools
''')
print("✅ company_agent/tools/base.py 已创建")

✅ company_agent/tools/base.py 已创建


### ✅ 验证：测试基类和注册表

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from company_agent.tools.base import BaseTool, ToolRegistry


# 创建一个临时测试工具
class HelloTool(BaseTool):
    @property
    def name(self):
        return "hello"

    @property
    def description(self):
        return "打个招呼"

    @property
    def parameters(self):
        return {"type": "object", "properties": {}, "required": []}

    async def __call__(self, **kwargs):
        return "Hello, World!"


# 注册并测试
registry = ToolRegistry()
registry.register(HelloTool())

print(f"注册表中有 {len(registry.get_all())} 个工具")
print(f"'hello' 工具存在: {'hello' in registry}")

import json
tool = registry.get("hello")
print(f"\n工具 'hello' 的 OpenAI 格式:")
print(json.dumps(tool.to_openai_tool(), indent=2, ensure_ascii=False))

注册表中有 1 个工具
'hello' 工具存在: True

工具 'hello' 的 OpenAI 格式:
{
  "type": "function",
  "function": {
    "name": "hello",
    "description": "打个招呼",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": []
    }
  }
}


---

## 🔧 Step 2: 编写第一个 API 工具

### 💡 核心概念：Function Calling 是如何工作的？

```text
1. 你将工具的 name, description, parameters 告诉 LLM
2. 用户提问 → LLM 分析问题 → 决定是否需要调用某个工具
3. 如果需要，LLM 返回 tool_calls: [{name: "xxx", arguments: {...}}]
4. 你根据 tool_calls 执行对应工具，把结果回传给 LLM
5. LLM 基于工具返回的结果生成最终答案
```

### 动手：编写 `tools/employees.py` — 员工查询工具

In [6]:
with open("company_agent/tools/employees.py", "w", encoding="utf-8") as f:
    f.write('''
"""员工查询工具 — 调用 /api/employees 获取员工信息"""
from typing import Any, Dict, Optional
import httpx
from .base import BaseTool


# 模拟员工数据（服务器不可用时使用）
MOCK_EMPLOYEES = {
    "all": [
        {"name": "张三", "department": "工程部", "position": "高级工程师"},
        {"name": "李四", "department": "工程部", "position": "前端开发"},
        {"name": "王五", "department": "产品部", "position": "产品经理"},
        {"name": "赵六", "department": "市场部", "position": "市场总监"},
        {"name": "钱七", "department": "人事部", "position": "HRBP"},
    ]
}


class GetEmployees(BaseTool):
    """查询公司员工信息"""

    def __init__(self, api_base_url: str = "http://localhost:8080"):
        self.api_base_url = api_base_url.rstrip("/")

    @property
    def name(self) -> str:
        return "get_employees"

    @property
    def description(self) -> str:
        return (
            "查询公司员工信息列表。可按部门筛选员工，"
            "返回员工姓名、部门、职位等信息。"
        )

    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "department": {
                    "type": "string",
                    "description": "可选，按部门筛选",
                    "enum": ["工程部", "产品部", "市场部", "人事部"],
                }
            },
            "required": [],
        }

    async def __call__(self, department: Optional[str] = None, **kwargs) -> str:
        params = {}
        if department:
            params["department"] = department

        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.api_base_url}/api/employees",
                    params=params,
                    timeout=5.0,
                )
                data = resp.json()
        except Exception:
            return self._mock_response(department)

        employees = data.get("employees", [])
        if not employees:
            return f"未找到{department or '所有'}部门的员工"

        lines = [f"共找到 {data.get('total', len(employees))} 名员工:"]
        for emp in employees:
            lines.append(f"  - {emp['name']} | {emp['department']} | {emp['position']}")
        return "\\n".join(lines)

    def _mock_response(self, department: Optional[str] = None) -> str:
        """服务器不可用时返回模拟数据"""
        if department:
            data = [e for e in MOCK_EMPLOYEES["all"] if e["department"] == department]
        else:
            data = MOCK_EMPLOYEES["all"]
        if not data:
            return f"未找到{department or '所有'}部门的员工"
        lines = [f"共找到 {len(data)} 名员工（模拟数据）:"]
        for emp in data:
            lines.append(f"  - {emp['name']} | {emp['department']} | {emp['position']}")
        return "\\n".join(lines)
''')
print("✅ company_agent/tools/employees.py 已创建")

✅ company_agent/tools/employees.py 已创建


In [7]:
with open("company_agent/tools/projects.py", "w", encoding="utf-8") as f:
    f.write('''
"""项目查询工具 — 调用 /api/projects 获取项目信息"""
from typing import Any, Dict, Optional
import httpx
from .base import BaseTool


# 模拟项目数据（服务器不可用时使用）
MOCK_PROJECTS = {
    "all": [
        {"id": "P001", "name": "智能客服平台", "status": "进行中", "progress": 65, "team": "工程部"},
        {"id": "P002", "name": "数据中台建设", "status": "进行中", "progress": 30, "team": "工程部"},
        {"id": "P003", "name": "官网改版", "status": "规划中", "progress": 0, "team": "产品部"},
        {"id": "P004", "name": "Q3 营销活动", "status": "规划中", "progress": 10, "team": "市场部"},
        {"id": "P005", "name": "员工培训系统", "status": "已完成", "progress": 100, "team": "人事部"},
    ]
}


class GetProjects(BaseTool):
    """查询公司项目列表"""

    def __init__(self, api_base_url: str = "http://localhost:8080"):
        self.api_base_url = api_base_url.rstrip("/")

    @property
    def name(self) -> str:
        return "get_projects"

    @property
    def description(self) -> str:
        return (
            "查询公司项目列表。可按状态筛选项目，"
            "返回项目名称、状态、进度、负责团队等信息。"
        )

    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "status": {
                    "type": "string",
                    "description": "可选，按状态筛选",
                    "enum": ["进行中", "规划中", "已完成"],
                }
            },
            "required": [],
        }

    async def __call__(self, status: Optional[str] = None, **kwargs) -> str:
        params = {}
        if status:
            params["status"] = status

        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.api_base_url}/api/projects",
                    params=params,
                    timeout=5.0,
                )
                data = resp.json()
        except Exception:
            return self._mock_response(status)

        projects = data.get("projects", [])
        if not projects:
            return f"未找到状态为'{status or '所有'}的项目"

        lines = [f"共找到 {data.get('total', len(projects))} 个项目:"]
        for proj in projects:
            lines.append(
                f"  - [{proj['id']}] {proj['name']} | {proj['status']} "
                f"| 进度{proj['progress']}% | {proj['team']}"
            )
        return "\\n".join(lines)

    def _mock_response(self, status: Optional[str] = None) -> str:
        """服务器不可用时返回模拟数据"""
        if status:
            data = [p for p in MOCK_PROJECTS["all"] if p["status"] == status]
        else:
            data = MOCK_PROJECTS["all"]
        if not data:
            return f"未找到状态为'{status or '所有'}的项目"
        lines = [f"共找到 {len(data)} 个项目（模拟数据）:"]
        for proj in data:
            lines.append(
                f"  - [{proj['id']}] {proj['name']} | {proj['status']} "
                f"| 进度{proj['progress']}% | {proj['team']}"
            )
        return "\\n".join(lines)
''')
print("✅ company_agent/tools/projects.py 已创建")

✅ company_agent/tools/projects.py 已创建


### ✅ 验证：注册工具并查看 OpenAI 格式

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from company_agent.tools import ToolRegistry
from company_agent.tools.employees import GetEmployees
from company_agent.tools.projects import GetProjects
import json

API_BASE_URL = "http://localhost:8080"
registry = ToolRegistry()
registry.register(GetEmployees(api_base_url=API_BASE_URL))
registry.register(GetProjects(api_base_url=API_BASE_URL))

print(f"✅ 已注册 {len(registry.get_all())} 个工具:\n")
for tool in registry.get_all():
    print(f"📝 {tool.name}: {tool.description[:50]}")

print("\n=== OpenAI Function Calling 格式 ===")
print(json.dumps(registry.to_openai_tools(), indent=2, ensure_ascii=False))

✅ 已注册 2 个工具:

📝 get_employees: 查询公司员工信息列表。可按部门筛选员工，返回员工姓名、部门、职位等信息。
📝 get_projects: 查询公司项目列表。可按状态筛选项目，返回项目名称、状态、进度、负责团队等信息。

=== OpenAI Function Calling 格式 ===
[
  {
    "type": "function",
    "function": {
      "name": "get_employees",
      "description": "查询公司员工信息列表。可按部门筛选员工，返回员工姓名、部门、职位等信息。",
      "parameters": {
        "type": "object",
        "properties": {
          "department": {
            "type": "string",
            "description": "可选，按部门筛选",
            "enum": [
              "工程部",
              "产品部",
              "市场部",
              "人事部"
            ]
          }
        },
        "required": []
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "get_projects",
      "description": "查询公司项目列表。可按状态筛选项目，返回项目名称、状态、进度、负责团队等信息。",
      "parameters": {
        "type": "object",
        "properties": {
          "status": {
            "type": "string",
            "description": "可选，按状态筛选",
            "enum": [
              "进行中",
   

> 🎯 **理解要点**：`to_openai_tools()` 输出的 JSON 就是告诉 LLM「你有哪些工具可用、每个工具需要什么参数」的关键数据。LLM 会根据这些描述决定调用哪个工具。

---

## 📝 Step 3: 提示词设计

### 💡 核心概念：提示词如何引导 Agent 行为

Agent 的「智能」来自两个部分：
1. **LLM 的能力**：理解问题、决定调用什么工具
2. **提示词**：告诉 LLM「你是谁」「你该怎么做」「格式要求」

好的提示词就像好的工作说明——越清晰具体，Agent 的表现越好。

### 动手：创建提示词文件

In [9]:
with open("company_agent/prompts/__init__.py", "w", encoding="utf-8") as f:
    f.write('''
"""prompts 包 — 从 .md 文件加载提示词"""
import os

_prompt_dir = os.path.dirname(os.path.abspath(__file__))

with open(os.path.join(_prompt_dir, "system.md"), "r", encoding="utf-8") as f_p:
    SYSTEM_PROMPT = f_p.read()

with open(os.path.join(_prompt_dir, "next_step.md"), "r", encoding="utf-8") as f_p:
    NEXT_STEP_PROMPT = f_p.read()


def build_next_step_prompt(user_query: str, search_history: str = "") -> str:
    """构建每一步的提示词"""
    prompt = NEXT_STEP_PROMPT.format(
        user_query=user_query,
        search_history=search_history.strip(),
    )
    return prompt


__all__ = ["SYSTEM_PROMPT", "NEXT_STEP_PROMPT", "build_next_step_prompt"]
''')
print("✅ company_agent/prompts/__init__.py 已创建")

✅ company_agent/prompts/__init__.py 已创建


In [10]:
with open("company_agent/prompts/system.md", "w", encoding="utf-8") as f:
    f.write('''
# 系统提示词

你是一个公司内部智能助手，可以调用公司内部 API 来回答员工的问题。

## 可用工具

- get_employees: 查询员工信息，可按部门筛选
- get_projects: 查询项目信息，可按状态筛选
- get_metrics: 获取公司关键业务指标
- search_company: 在公司内部知识库中搜索信息

## 工作流程

1. **分析**: 理解用户问题，判断需要什么信息
2. **决策**: 选择合适的工具（API）来获取信息
3. **综合**: 基于获取的信息，生成完整的回答

## 重要原则

- 优先使用具体的查询工具（如 get_employees），而不是搜索工具
- 当需要多个维度的信息时，可以并行调用多个工具
- 回答时使用 Markdown 格式，让结果清晰易读
- 如果工具返回的信息不足以回答问题，说明局限性
''')
print("✅ company_agent/prompts/system.md 已创建")

✅ company_agent/prompts/system.md 已创建


In [11]:
with open("company_agent/prompts/next_step.md", "w", encoding="utf-8") as f:
    f.write('''
请帮助用户回答问题。以下是用户的原始问题：

## 用户问题

{user_query}

{search_history}

## 你的任务

根据上面的问题，如果你已经收集了足够的信息，可以直接回答用户；
否则，请调用合适的工具来获取更多信息。
''')
print("✅ company_agent/prompts/next_step.md 已创建")

✅ company_agent/prompts/next_step.md 已创建


### ✅ 验证：查看提示词加载效果

In [12]:
from company_agent.prompts import SYSTEM_PROMPT, build_next_step_prompt

print("=== 系统提示词 ===")
print(SYSTEM_PROMPT)

print("\n=== 动态构建的提示词（有历史） ===")
prompt = build_next_step_prompt(
    "工程部有多少人？",
    search_history="已查询：get_employees 返回 2 名员工",
)
print(prompt)

=== 系统提示词 ===

# 系统提示词

你是一个公司内部智能助手，可以调用公司内部 API 来回答员工的问题。

## 可用工具

- get_employees: 查询员工信息，可按部门筛选
- get_projects: 查询项目信息，可按状态筛选
- get_metrics: 获取公司关键业务指标
- search_company: 在公司内部知识库中搜索信息

## 工作流程

1. **分析**: 理解用户问题，判断需要什么信息
2. **决策**: 选择合适的工具（API）来获取信息
3. **综合**: 基于获取的信息，生成完整的回答

## 重要原则

- 优先使用具体的查询工具（如 get_employees），而不是搜索工具
- 当需要多个维度的信息时，可以并行调用多个工具
- 回答时使用 Markdown 格式，让结果清晰易读
- 如果工具返回的信息不足以回答问题，说明局限性


=== 动态构建的提示词（有历史） ===

请帮助用户回答问题。以下是用户的原始问题：

## 用户问题

工程部有多少人？

已查询：get_employees 返回 2 名员工

## 你的任务

根据上面的问题，如果你已经收集了足够的信息，可以直接回答用户；
否则，请调用合适的工具来获取更多信息。



> 🎯 **理解要点**：为什么用 `.md` 文件而非硬编码在 Python 中？
> - 非技术人员可以直接修改提示词
> - 版本控制清晰（git diff 可以看到改了哪些提示词）
> - 支持多语言（system_en.md, system_zh.md）

---

## 🧠 Step 4: LLM 客户端封装

### 💡 核心概念：OpenAI 兼容协议

大多数 LLM 服务商都提供 **OpenAI 兼容的 API**。这意味着：
- 使用 `openai` Python SDK（`AsyncOpenAI` 类）
- 只需改 `base_url` 和 `api_key`，无需修改调用代码
- 函数调用（Function Calling）的格式也是统一的

### 动手：编写 `llm/client.py`

In [13]:
with open("company_agent/llm/__init__.py", "w", encoding="utf-8") as f:
    f.write('''
"""llm 包"""
from .client import LLMClient

__all__ = ["LLMClient"]
''')
print("✅ company_agent/llm/__init__.py 已创建")

✅ company_agent/llm/__init__.py 已创建


In [14]:
with open("company_agent/llm/client.py", "w", encoding="utf-8") as f:
    f.write('''
"""LLM 客户端 — 使用 OpenAI 兼容协议
支持真实 API 和模拟模式（无 API Key 时自动降级）"""
import json
import os
from typing import Any, Dict, List, Optional, Tuple


class LLMClient:
    """LLM 客户端"""

    BASE_URL = "https://token-plan.cn-beijing.maas.aliyuncs.com/compatible-mode/v1"
    MODEL = "deepseek-v4-flash"

    def __init__(
        self,
        api_key: Optional[str] = None,
        model: Optional[str] = None,
        base_url: Optional[str] = None,
    ):
        # 优先级：构造函数参数 > 环境变量 > 空字符串（模拟模式）
        self.api_key = api_key or os.environ.get("DEEPSEEK_API_KEY", "")
        self.model = model or self.MODEL
        self.base_url = base_url or self.BASE_URL
        self._use_mock = not self.api_key

    @property
    def mode(self) -> str:
        return "🧪 模拟模式" if self._use_mock else "🔌 真实 API 模式"

    async def chat(
        self,
        messages: List[Dict[str, str]],
        tools: Optional[List[Dict[str, Any]]] = None,
    ) -> Tuple[str, List[Dict]]:
        """
        与 LLM 对话。

        Args:
            messages: 对话历史 [{role, content}, ...]
            tools: 工具定义列表（OpenAI 格式）

        Returns:
            (回复文本, tool_calls 列表)
            - 如果 LLM 决定调用工具，text 通常为空，tool_calls 有内容
            - 如果 LLM 直接回答，text 有内容，tool_calls 为空
        """
        if self._use_mock:
            return self._mock_chat(messages)

        try:
            from openai import AsyncOpenAI
            client = AsyncOpenAI(api_key=self.api_key, base_url=self.base_url)

            kwargs = {"model": self.model, "messages": messages}
            if tools:
                kwargs["tools"] = tools

            response = await client.chat.completions.create(**kwargs)

            choice = response.choices[0]
            message = choice.message

            tool_calls = []
            if message.tool_calls:
                for tc in message.tool_calls:
                    tool_calls.append({
                        "id": tc.id,
                        "name": tc.function.name,
                        "arguments": json.loads(tc.function.arguments),
                    })

            return message.content or "", tool_calls

        except Exception as e:
            print(f"⚠️ LLM API 错误: {e}，切换到模拟模式")
            self._use_mock = True
            return self._mock_chat(messages)

    def _mock_chat(self, messages: List[Dict]) -> Tuple[str, List[Dict]]:
        """模拟 LLM 响应（用于无 API Key 时）"""
        last_msg = messages[-1].get("content", "")
        has_results = "已获取" in last_msg or "结果" in last_msg

        if has_results:
            return (
                "根据查询到的信息，我来为您总结：\\n\\n"
                "这是模拟 LLM 生成的回答。在实际使用时，DeepSeek-V4-Flash "
                "会分析真实的 API 结果并生成更全面的回答。"
            ), []
        else:
            return "", [
                {"id": "call_mock_1", "name": "get_employees", "arguments": {}},
                {
                    "id": "call_mock_2",
                    "name": "get_projects",
                    "arguments": {"status": "进行中"},
                },
            ]
''')
print("✅ company_agent/llm/client.py 已创建")

✅ company_agent/llm/client.py 已创建


### ✅ 验证：测试 LLM 调用

In [ ]:
import os
# 设置 API Key（也可通过环境变量 DEEPSEEK_API_KEY 设置）
os.environ["DEEPSEEK_API_KEY"] = "your-api-key-here"

import sys, os as _os
sys.path.insert(0, _os.path.abspath("."))

from company_agent.llm import LLMClient
from company_agent.tools import ToolRegistry
from company_agent.tools.employees import GetEmployees
from company_agent.tools.projects import GetProjects

llm = LLMClient()
print(f"模型: {llm.model}")
print(f"端点: {llm.base_url}")
print(f"模式: {llm.mode}")

registry = ToolRegistry()
registry.register(GetEmployees())
registry.register(GetProjects())

messages = [
    {"role": "system", "content": "你是公司助手。"},
    {"role": "user", "content": "工程部有哪些员工？"},
]
tools = registry.to_openai_tools()

text, tool_calls = await llm.chat(messages, tools=tools)

print(f"\n{'='*50}")
if tool_calls:
    print(f"🔧 LLM 决定调用 {len(tool_calls)} 个工具:")
    for tc in tool_calls:
        print(f"   - {tc['name']}({tc['arguments']})")
else:
    print(f"💬 LLM 直接回答: {text[:100]}...")
print(f"{'='*50}")

模型: deepseek-v4-flash
端点: https://token-plan.cn-beijing.maas.aliyuncs.com/compatible-mode/v1
模式: 🔌 真实 API 模式

🔧 LLM 决定调用 1 个工具:
   - get_employees({'department': '工程部'})


> 🎯 **理解要点**：`llm.chat()` 的返回值是 `(text, tool_calls)`：
> - `tool_calls` 有内容 → LLM 需要你执行工具，把结果回传
> - `tool_calls` 为空 → LLM 已经给出了最终答案
> 这就是 Agent 推理循环的判断条件！

---

## 🔄 Step 5: Agent 推理循环（核心！）

### 💡 核心概念：ReAct 模式

**ReAct** = **Re**asoning + **Act**ing，是目前最主流的 Agent 架构：

```
第 1 轮：思考(分析用户问题) → 行动(调用工具) → 观察(获取工具结果)
第 2 轮：思考(分析工具结果) → 行动(再调用工具 或 给出答案)
第 3 轮：... 直到给出最终答案
```

每轮的决策都是 **LLM 做出的**，你只需要提供：
1. 工具列表（LLM 知道有哪些工具可用）
2. 提示词（LLM 知道该怎么做）
3. 执行工具并返回结果（LLM 知道观察到了什么）

### 并行执行的优势

当 LLM 决定同时调用多个工具时（比如同时查员工和项目），我们可以
用 `asyncio.gather()` **并行执行**，而不是串行等待，大幅提升响应速度。

### 动手：编写 `agent/agent.py`

In [16]:
with open("company_agent/agent/__init__.py", "w", encoding="utf-8") as f:
    f.write('''
"""agent 包"""
from .agent import AgentWithTools

__all__ = ["AgentWithTools"]
''')
print("✅ company_agent/agent/__init__.py 已创建")

✅ company_agent/agent/__init__.py 已创建


In [17]:
with open("company_agent/agent/agent.py", "w", encoding="utf-8") as f:
    f.write('''
"""Agent 推理循环 — ReAct 模式实现"""
import json
import time
import asyncio
from typing import List, Dict

from ..llm import LLMClient
from ..tools import ToolRegistry
from ..prompts import SYSTEM_PROMPT, build_next_step_prompt


class AgentWithTools:
    """Agent 推理循环"""

    def __init__(
        self,
        llm: LLMClient,
        registry: ToolRegistry,
        system_prompt: str = SYSTEM_PROMPT,
        max_steps: int = 5,
    ):
        self.llm = llm
        self.registry = registry
        self.system_prompt = system_prompt
        self.max_steps = max_steps

    async def run(self, user_query: str) -> str:
        """
        执行 Agent 推理循环。

        流程：
        1. 构建提示词，调用 LLM
        2. LLM 返回 tool_calls → 并行执行工具 → 结果加入对话历史
        3. 重复直到 LLM 给出最终答案或达到最大步骤
        """
        print(f"\\n{'='*60}")
        print(f"🤖 Agent 启动")
        print(f"   问题: {user_query}")
        print(f"{'='*60}\\n")

        # 初始化对话历史
        messages = [{"role": "system", "content": self.system_prompt}]
        search_history = ""

        for step in range(1, self.max_steps + 1):
            print(f"\\n{'─'*50}")
            print(f"📍 第 {step}/{self.max_steps} 步")
            print(f"{'─'*50}")

            # 1. 构建当前轮的提示词
            user_prompt = build_next_step_prompt(user_query, search_history)
            messages.append({"role": "user", "content": user_prompt})

            # 2. 调用 LLM
            print(f"   🧠 调用 LLM 决策...")
            response_text, tool_calls = await self.llm.chat(
                messages=messages,
                tools=self.registry.to_openai_tools(),
            )

            # 3. 判断：LLM 是否调用了工具？
            if not tool_calls:
                # 没有 tool_calls → LLM 给出了最终答案
                print(f"   ✅ LLM 返回了最终答案")
                print(f"{'='*60}")
                print(f"💬 {response_text or '未生成回答'}")
                print(f"{'='*60}")
                return response_text or "未生成回答"

            # 4. LLM 决定调用工具 → 打印决策
            print(f"   🔧 LLM 决定调用 {len(tool_calls)} 个工具:")
            for tc in tool_calls:
                args = json.dumps(tc["arguments"], ensure_ascii=False)
                print(f"      - {tc['name']}({args})")

            # 5. 并行执行所有工具
            results = await self._execute_tools_parallel(tool_calls)

            # 6. 将工具结果加入对话历史（LLM 下一轮会看到这些结果）
            for result in results:
                messages.append({
                    "role": "tool",
                    "tool_call_id": result["tool_call_id"],
                    "content": result["result"],
                })
                search_history += f"\\n### {result['name']}\\n"
                search_history += f"结果: {result['result'][:200]}\\n"

        print(f"\\n⚠️ 达到最大步骤数 ({self.max_steps})，未能生成回答")
        return "已达到最大推理步骤。"

    async def _execute_tools_parallel(self, tool_calls: List[Dict]) -> List[Dict]:
        """并行执行多个工具调用"""
        if not tool_calls:
            return []

        async def execute_one(tc: Dict) -> Dict:
            tool_name = tc["name"]
            args = tc.get("arguments", {})

            tool = self.registry.get(tool_name)
            if not tool:
                return {
                    "tool_call_id": tc.get("id", ""),
                    "name": tool_name,
                    "result": f"未知工具: {tool_name}",
                }

            print(f"   🚀 执行: {tool_name}({json.dumps(args, ensure_ascii=False)})")
            result = await tool(**args)
            return {
                "tool_call_id": tc.get("id", ""),
                "name": tool_name,
                "result": result,
            }

        print(f"   ⚡ 并行执行 {len(tool_calls)} 个工具调用...")
        start_time = time.time()

        # 关键：asyncio.gather 让多个工具同时运行
        tasks = [execute_one(tc) for tc in tool_calls]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        elapsed = time.time() - start_time
        print(f"   ✅ 全部完成，耗时 {elapsed:.2f}s")

        processed_results = []
        for i, result in enumerate(results):
            if isinstance(result, Exception):
                processed_results.append({
                    "tool_call_id": tool_calls[i].get("id", ""),
                    "name": tool_calls[i]["name"],
                    "result": f"执行错误: {str(result)}",
                })
            else:
                processed_results.append(result)

        return processed_results
''')
print("✅ company_agent/agent/agent.py 已创建")

✅ company_agent/agent/agent.py 已创建


---

## 🎬 Step 6: 补齐剩余工具 + 运行入口

为了让 Agent 拥有完整能力，我们还需要添加 `GetMetrics` 和 `SearchCompany` 两个工具。
最后创建 `main.py` 作为统一入口。

In [18]:
with open("company_agent/tools/metrics.py", "w", encoding="utf-8") as f:
    f.write('''
"""公司指标工具 — 调用 /api/metrics"""
from typing import Any, Dict
import httpx
from .base import BaseTool


# 模拟指标数据（服务器不可用时使用）
MOCK_METRICS = {
    "metrics": {
        "Q3 营收": "¥1,280 万",
        "员工总数": "187 人",
        "满意度评分": "4.6/5.0",
        "进行中项目": "5 个",
        "本月入职": "12 人",
    }
}


class GetMetrics(BaseTool):
    """获取公司关键业务指标"""

    def __init__(self, api_base_url: str = "http://localhost:8080"):
        self.api_base_url = api_base_url.rstrip("/")

    @property
    def name(self) -> str:
        return "get_metrics"

    @property
    def description(self) -> str:
        return "获取公司关键业务指标，包括营收、员工数、满意度等"

    @property
    def parameters(self) -> Dict[str, Any]:
        return {"type": "object", "properties": {}, "required": []}

    async def __call__(self, **kwargs) -> str:
        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.api_base_url}/api/metrics",
                    timeout=5.0,
                )
                data = resp.json()
        except Exception:
            return self._mock_response()

        lines = ["📊 公司关键指标:"]
        for key, val in data.get("metrics", {}).items():
            lines.append(f"  - {key}: {val}")
        return "\\n".join(lines)

    def _mock_response(self) -> str:
        """服务器不可用时返回模拟数据"""
        lines = ["📊 公司关键指标（模拟数据）:"]
        for key, val in MOCK_METRICS["metrics"].items():
            lines.append(f"  - {key}: {val}")
        return "\\n".join(lines)
''')
print("✅ company_agent/tools/metrics.py 已创建")

✅ company_agent/tools/metrics.py 已创建


In [19]:
with open("company_agent/tools/search.py", "w", encoding="utf-8") as f:
    f.write('''
"""内部搜索工具 — 调用 /api/search"""
from typing import Any, Dict
import httpx
from .base import BaseTool


# 模拟搜索数据（服务器不可用时使用）
MOCK_SEARCH = {
    "AI": [
        {"title": "AI 战略规划文档", "snippet": "公司将在 Q4 启动 AI 中台建设，重点投入 NLP 和推荐系统方向..."},
        {"title": "智能客服项目立项", "snippet": "基于大语言模型的智能客服项目已获批准，预计年底上线..."},
    ],
    "默认": [
        {"title": "公司管理制度 V3.2", "snippet": "本制度适用于全体员工，涵盖考勤、福利、绩效考核等方面..."},
        {"title": "季度 OKR 指引", "snippet": "请各部门在每月初提交本季度 OKR，由管理层评审后发布..."},
    ],
}


class SearchCompany(BaseTool):
    """在公司内部知识库中搜索"""

    def __init__(self, api_base_url: str = "http://localhost:8080"):
        self.api_base_url = api_base_url.rstrip("/")

    @property
    def name(self) -> str:
        return "search_company"

    @property
    def description(self) -> str:
        return (
            "在公司内部知识库中搜索信息。当无法通过结构化 API "
            "获取信息时使用此工具。返回相关文档摘要。"
        )

    @property
    def parameters(self) -> Dict[str, Any]:
        return {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "搜索关键词"},
            },
            "required": ["query"],
        }

    async def __call__(self, query: str, **kwargs) -> str:
        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(
                    f"{self.api_base_url}/api/search",
                    params={"q": query},
                    timeout=5.0,
                )
                data = resp.json()
        except Exception:
            return self._mock_response(query)

        results = data.get("results", [])
        if not results:
            return f"未找到关于 '{query}' 的信息"

        lines = [f"找到 {len(results)} 条关于 '{query}' 的结果:"]
        for r in results:
            lines.append(
                f"  - {r.get('title', '无标题')}: "
                f"{r.get('snippet', '')[:100]}"
            )
        return "\\n".join(lines)

    def _mock_response(self, query: str) -> str:
        """服务器不可用时返回模拟数据"""
        # 尝试匹配关键词
        for keyword, results in MOCK_SEARCH.items():
            if keyword != "默认" and keyword.lower() in query.lower():
                break
        else:
            results = MOCK_SEARCH["默认"]
        lines = [f"找到 {len(results)} 条关于 '{query}' 的结果（模拟数据）:"]
        for r in results:
            lines.append(f"  - {r.get('title', '无标题')}: {r.get('snippet', '')[:100]}")
        return "\\n".join(lines)
''')
print("✅ company_agent/tools/search.py 已创建")

✅ company_agent/tools/search.py 已创建


In [20]:
with open("company_agent/main.py", "w", encoding="utf-8") as f:
    f.write('''
"""主运行入口 — 提供便捷的 run() 函数"""
import os
import asyncio
from typing import Optional

from .tools import ToolRegistry
from .tools.employees import GetEmployees
from .tools.projects import GetProjects
from .tools.metrics import GetMetrics
from .tools.search import SearchCompany
from .llm import LLMClient
from .agent import AgentWithTools


def create_registry(api_base_url: str = "http://localhost:8080") -> ToolRegistry:
    """创建并注册所有工具"""
    registry = ToolRegistry()
    registry.register(GetEmployees(api_base_url=api_base_url))
    registry.register(GetProjects(api_base_url=api_base_url))
    registry.register(GetMetrics(api_base_url=api_base_url))
    registry.register(SearchCompany(api_base_url=api_base_url))
    return registry


async def run(
    query: str,
    api_key: Optional[str] = None,
    api_base_url: str = "http://localhost:8080",
    max_steps: int = 3,
) -> str:
    """运行 Agent 的便捷函数"""
    registry = create_registry(api_base_url=api_base_url)
    llm = LLMClient(api_key=api_key)
    agent = AgentWithTools(llm=llm, registry=registry, max_steps=max_steps)
    return await agent.run(query)


if __name__ == "__main__":
    asyncio.run(run("我们公司有哪些进行中的项目？工程部有哪些员工？"))
''')
print("✅ company_agent/main.py 已创建")

✅ company_agent/main.py 已创建


---

### 6.1 运行完整 Agent

现在万事齐备！让我们运行完整的 Agent 推理循环：

In [ ]:
import os
# 设置 API Key（也可通过环境变量 DEEPSEEK_API_KEY 设置）
os.environ["DEEPSEEK_API_KEY"] = "your-api-key-here"

import sys, os as _os
sys.path.insert(0, _os.path.abspath("."))

from company_agent.main import run

# 示例 1：多工具查询
query1 = "我们公司有哪些进行中的项目？工程部有哪些员工？"
result1 = await run(query1, api_base_url="http://localhost:8080", max_steps=3)
print(f"\n✅ 示例 1 完成！")


🤖 Agent 启动
   问题: 我们公司有哪些进行中的项目？工程部有哪些员工？


──────────────────────────────────────────────────
📍 第 1/3 步
──────────────────────────────────────────────────
   🧠 调用 LLM 决策...
   🔧 LLM 决定调用 2 个工具:
      - get_projects({"status": "进行中"})
      - get_employees({"department": "工程部"})
   ⚡ 并行执行 2 个工具调用...
   🚀 执行: get_projects({"status": "进行中"})
   🚀 执行: get_employees({"department": "工程部"})
   ✅ 全部完成，耗时 3.28s

──────────────────────────────────────────────────
📍 第 2/3 步
──────────────────────────────────────────────────
   🧠 调用 LLM 决策...
   ✅ LLM 返回了最终答案
💬 好的，我已经通过调用两个工具获取到了相关信息，现在为你汇总回答：

---

## 📋 公司当前进行中的项目

| 项目编号 | 项目名称 | 状态 | 进度 | 负责团队 |
| :--- | :--- | :--- | :--- | :--- |
| P001 | **智能客服平台** | ✅ 进行中 | 65% | 工程部 |
| P002 | **数据中台建设** | ✅ 进行中 | 30% | 工程部 |

---

## 👥 工程部员工列表

| 姓名 | 部门 | 职位 |
| :--- | :--- | :--- |
| **张三** | 工程部 | 高级工程师 |
| **李四** | 工程部 | 前端开发 |

---

目前公司共有 **2个进行中的项目**（均由工程部负责），以及 **2名工程部员工**。如果你需要了解更多细节（例如每个项目的具体成员、员工联系方式等），可以进一步查询。

✅ 示例 1 完成！


In [ ]:
# 示例 2：指标查询
query2 = "公司 Q3 的营收情况如何？现在有哪些规划中的项目？"
result2 = await run(query2, api_base_url="http://localhost:8080", max_steps=3)
print(f"\n✅ 示例 2 完成！")

In [ ]:
# 示例 3：搜索查询
query3 = "公司关于 AI 的项目有哪些？"
result3 = await run(query3, api_base_url="http://localhost:8080", max_steps=3)
print(f"\n✅ 示例 3 完成！")

---

## 🎓 扩展练习

现在你已经从零构建了一个完整的 Agent 项目。试试完成以下练习来加深理解：

### 练习 1：添加 `GetDepartments` 工具

在 `company_agent/tools/` 目录下创建 `departments.py`，实现一个查询所有部门列表的工具：

```python
# 提示：
# 1. 继承 BaseTool
# 2. 实现 name = "get_departments"
# 3. description 描述清楚功能
# 4. parameters 不需要参数（空 properties）
# 5. __call__ 中调用 http://localhost:8080/api/departments
# 6. 最后在 main.py 的 create_registry() 中注册
```

### 练习 2：添加错误重试

在工具的 `__call__` 方法中，如果 HTTP 请求失败，自动重试 3 次：

```python
async def __call__(self, **kwargs) -> str:
    for attempt in range(3):
        try:
            async with httpx.AsyncClient() as client:
                resp = await client.get(..., timeout=5.0)
                return str(resp.json())
        except Exception as e:
            if attempt == 2:
                return f"API 调用失败（已重试 3 次）: {e}"
            await asyncio.sleep(1)
```

### 练习 3：添加日志系统

在 Agent 的 `run()` 方法中添加 `logging`，记录每一步的操作和时间。

---

## 📋 总结：如何扩展你的 Agent

### 添加新 API 工具（3 步）

**1. 在 `tools/` 目录下创建新文件**
```python
# tools/departments.py
from .base import BaseTool

class GetDepartments(BaseTool):
    @property
    def name(self): return "get_departments"
    @property
    def description(self): return "查询公司所有部门列表"
    @property
    def parameters(self):
        return {"type": "object", "properties": {}, "required": []}
    async def __call__(self, **kwargs) -> str:
        import httpx
        async with httpx.AsyncClient() as c:
            r = await c.get(f"{self.api_base_url}/api/departments")
            return str(r.json())
```

**2. 在 `main.py` 的 `create_registry()` 中注册**
```python
from .tools.departments import GetDepartments

def create_registry(api_base_url):
    registry = ToolRegistry()
    registry.register(GetDepartments(api_base_url=api_base_url))  # ← 新增
    # ... 其他工具
    return registry
```

**3. 完成！** LLM 会自动发现新工具并决定是否需要调用它。

### 切换 LLM 模型

```python
# 方法 1：修改 llm/client.py 中的常量
BASE_URL = "https://your-api-endpoint/v1"
MODEL = "your-model-name"

# 方法 2：运行时指定
llm = LLMClient(
    api_key="your-key",
    model="gpt-4o",
    base_url="https://api.openai.com/v1"
)
```

### 修改提示词

直接编辑 `prompts/system.md` 或 `prompts/next_step.md`，无需修改 Python 代码。

---

## 🔗 后续学习

| 方向 | 推荐资源 |
|------|----------|
| Agent 架构论文 | [ReAct: Synergizing Reasoning and Acting](https://arxiv.org/abs/2210.03629) |
| Function Calling | [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling) |
| 生产实践 | 添加认证、限流、监控、单元测试 |
| 更多工具 | LangChain Tools, MCP (Model Context Protocol) |